# AeroCast — Notebook 02: Feature Engineering and Chronological Data Splitting

## Objectives:
This notebook prepares the final feature-engineered dataset for all four machine learning models:
1. **Temporal & Cyclical Features**: Hour, Day, Month, Day-of-week, Weekend indicator, and sine/cosine cyclical encodings.
2. **Historical Lag Features**: Lags at 1, 2, 3, 6, 12, and 24 hours for all 7 pollutants and CPCB AQI.
3. **Rolling Statistics**: 3h, 6h, 12h, and 24h rolling means and rolling standard deviations using past observations only.
4. **Target Creation**: Next-hour AQI target (`Target_AQI` = AQI(t+1)) and next-hour pollutant concentration targets (`Target_PM2.5`, `Target_PM10`, `Target_NO2`, etc.).
5. **Data Cleaning**: Short gap interpolation (limit=3 hours); never fill long gaps.
6. **Chronological Splitting**: 70% Training, 15% Validation, 15% Testing. Zero temporal leakage.


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))

from src.preprocessing.feature_engineering import load_and_pivot_raw, build_features, split_and_save

## 1. Load Hourly Pivoted Observations & Calculate CPCB AQI

In [ ]:
df_pivot = load_and_pivot_raw('../data/raw/site_113_raw.csv')
print(f"Pivoted Hourly Shape: {df_pivot.shape}")
print(f"Date Span: {df_pivot.index.min()} to {df_pivot.index.max()}")

## 2. Execute Feature Engineering Pipeline
We compute temporal, lag, rolling, and target variables strictly using past and current observations.

In [ ]:
clean_df = build_features(df_pivot)
print(f"Clean Feature Matrix Shape: {clean_df.shape}")

## 3. Chronological Data Splitting (70% Train, 15% Val, 15% Test)
Time-series datasets must NEVER be randomly shuffled. Chronological splitting preserves temporal order.

In [ ]:
train_df, val_df, test_df, feature_cols = split_and_save(clean_df, output_dir='../data/processed')

## 4. Verification of Processed Splits & Schema Consistency

In [ ]:
print("Split Summary:")
print(f"  Training:   {len(train_df):,} rows ({train_df.index.min()} to {train_df.index.max()})")
print(f"  Validation: {len(val_df):,} rows ({val_df.index.min()} to {val_df.index.max()})")
print(f"  Testing:    {len(test_df):,} rows ({test_df.index.min()} to {test_df.index.max()})")
print(f"\nTotal Input Features: {len(feature_cols)}")
print(f"Sample Features: {feature_cols[:10]}")

In [ ]:
# Verify zero NaNs in targets
target_cols = [c for c in clean_df.columns if c.startswith('Target_') and not c.endswith('_Category')]
print("Target NaN check across splits:")
print("Train target NaNs:", train_df[target_cols].isna().sum().sum())
print("Val target NaNs:  ", val_df[target_cols].isna().sum().sum())
print("Test target NaNs: ", test_df[target_cols].isna().sum().sum())

## 5. Summary & Hand-off to Model Notebooks
The processed splits have been saved to:
- `data/processed/train.csv` (11,422 rows)
- `data/processed/val.csv` (2,447 rows)
- `data/processed/test.csv` (2,449 rows)
- `data/processed/feature_metadata.json` (119 features)

All subsequent model notebooks (Linear Regression, Random Forest, XGBoost, and ANN) ingest these exact splits.
